In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window 
# Work in UTC so Spark never shifts times for UK summer time
spark.conf.set("spark.sql.session.timeZone", "UTC")

# load the table saved previously in 01_load
df = spark.table("elec_stacked")
df.columns

## 1. All columns that need to be cleaned:
- settlement_date, settlement_period - time uniformity required 

- NFS (note for self) - nullable = True, Allowed / may contain nulls
- printSchema() - funciton which returns the structure of teh dataframe (or selected col. in this case) as a visual tree format)

In [0]:
df.select("settlement_date", "settlement_period").printSchema()
display(df.select("settlement_date", "settlement_period").limit(5))
# the dates are shown in half hour time incremenets 1-48 = 1 day 
# date is shown as text, it needs to be converte to datetime before isolation of individual date and time components. 

In [0]:
# turn text into date format 
# (2023 rows use a 2-digit year, so try both formats)
df = df.withColumn( # replace a column ( df["col"] = ... in pandas)
    "settlement_date", 
    F.coalesce( # use the FIRST of the options below that isn't blank
        F.expr( # run SQL inside PySpark
            "try_to_date(settlement_date, 'dd-MMM-yyyy')"  # iteration 1: '01-JAN-2021' style (4-digit year)
        ), #  if the text doesn't fit, gives blank instead of crashing
        F.expr(
            "try_to_date(settlement_date, 'dd-MMM-yy')" # option 2: '01-Jan-23' style (2-digit year)
        ) # only used when option 1 gave blank
    )
)

# python then SQL with min, max (aggeregate funcitons) with alias : AS
# ensure that it worked across 2018-2022
df.select(
    F.min("settlement_date"),
    F.max("settlement_date"),
    F.count(F.when(F.col("settlement_date").isNull(), 1)).alias("bad_dates")
).show()
# GOAL - bad dates = 0


In [0]:
df = df.drop("embedded_wind_capacity", "embedded_solar_capacity")


- Build a map that sorts the rows into time order per day, then sort and state half hour time incrememnts into actual time 

In [0]:
# put rows into time order by date then my time period
w = Window.orderBy("settlement_date", "settlement_period")

# adds another row called row num = number the rows 1,2,3,4 ... into that order 
df = df.withColumn("row_num", F.row_number().over(w))

# create time stamp = start time  + row number - 1 x 30 minutes 
start = 1514764800                             #2018-01-01 00:00 GMT, written as seconds since 1970

df = df.withColumn("timestamp", F.timestamp_seconds(F.lit(start) + (F.col("row_num")-1)*1800))
df = df.drop("row_num")

df.select(
    F.min("timestamp"),
    F.max("timestamp"), 
    F.countDistinct("timestamp")).show() # should be 100% unique

Checks in SQL 

In [0]:
counts = df.groupBy("settlement_date").count()
counts.filter(counts["count"] != 48).count()
# 12 rows do not show 48 values that is fine - these acoomodate for time shifts and clocks change dates 

# how many unique pairs of date and half our increments are there, it should only ever occur once for each otherwise there are
unique_counts = df.groupBy(["settlement_date", "settlement_period"]).count()
unique_counts.filter(unique_counts["count"] != 1).count() # 0


Count the number of missing values:
- COUNT(*) counts every row, blanks included.
- COUNT(column) counts only the rows where that column has a value. It skips nulls.

In [0]:
# count the number of blanks or missing values there are in ech of these columns:
cols =  df.select(["scottish_transfer", "nsl_flow", "eleclink_flow", "viking_flow", "greenlink_flow"]).summary("count").show()

- Find rows where england_wales_demand is 0 = this shows missing data, MCAR
- failed to record input despite others having values 


In [0]:
display(
    df.filter(F.col("england_wales_demand") == 0).select("timestamp", "settlement_date", "settlement_period", "nd", "tsd", "england_wales_demand")
)
# 3 rows in 2018 show missing values for england_wales_demand - erronous, but as long as ND AND TSD are present it should be alright * note for later to recheck if needed*

- save the clean table

In [0]:
# save and add days , months, years, weekday etc, into the file so that it is all set up for the modelling 
local = F.from_utc_timestamp("timestamp", "Europe/London")    # UK clock time

df = (df
      .withColumn("year", F.year("timestamp"))
      .withColumn("month", F.month("timestamp"))
      .withColumn("weekday", (F.dayofweek(local) + 5) % 7)
      .withColumn("half_hour_intervals", F.hour(local) * 2 + F.floor(F.minute(local) / 30)))

display(df.select("timestamp", "year", "month", "weekday", "half_hour_intervals").limit(10))

Save clean table

In [0]:
(df.write
   .mode("overwrite")
   .option("overwriteSchema", "true") # allow the new columns
   .saveAsTable("elec_fin"))

spark.table("elec_fin").printSchema()
#printSchema() reads the table back, so you can check the save worked.
